# Turnkey: build your own detector
Create an external detector file and run it without changing Turnkey's registry.

In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

workspace = Path(tempfile.mkdtemp(prefix='turnkey-detector-'))
detector_path = workspace / 'length_detector.py'
detector_path.write_text('''from dataclasses import dataclass

from turnkey.components.detectors.base import Detector
from turnkey.schema import DetectorDecision, Sample

@dataclass(frozen=True)
class LengthDetector(Detector):
    threshold: int = 80

    def decide(self, sample: Sample) -> DetectorDecision:
        score = min(len(sample.prompt) / self.threshold, 1.0)
        return DetectorDecision(block=len(sample.prompt) >= self.threshold, score=score, reason="prompt_length")

def build():
    detector = LengthDetector()
    return detector.component(name="length-detector")
''')
completed = subprocess.run([sys.executable, '-m', 'turnkey.cli', 'dev', f'{detector_path}:build', '--out-dir', str(workspace / 'outputs')], check=True, capture_output=True, text=True)
run_dir = Path(completed.stdout.strip().splitlines()[-1])
run_dir

In [ ]:
run = json.loads((run_dir / 'run.json').read_text())
component = run['components']['intervention']
assert component['name'] == 'length-detector'
assert component['parameters']['threshold'] == 80
component